# Filtering GenVideo-Real (can use other real datasets as well)

GenVideo-Real contains many videos from shows, etc. -- many cuts -- need to minimize them
Also remove videos with cartoons, illustrations, etc

In [ ]:
!pip install -q huggingface_hub tqdm decord torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 159.9 MB/s eta 0:00:00


In [ ]:
from huggingface_hub import login
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
import shutil
from pathlib import Path

import numpy as np
import torch
from decord import VideoReader, gpu
from huggingface_hub import snapshot_download, upload_folder

REPO_ID = "ironiss/PhysVidDetect-v1"
FOLDERS = ["real/GenVideo-Real", "real/GenVideo-Real-additional-3k"] #two parts, because I downloaded not enough video at first
VIDEO_EXTS = {".mp4", ".mov", ".webm", ".mkv", ".avi"}
TARGET = 4000
UPLOAD_PATH = "real/GenVideo-Real-clean-3k"

SAMPLE_FRAMES = 30
CUT_THRESHOLD = 0.4
MIN_FRAMES = 16
HIST_BINS = 32

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dec_ctx = gpu(0)
print(f"Device: {device}, Decord: {'GPU' if torch.cuda.is_available() else 'CPU'}")

WORK_DIR = Path("/content/genvideo_filter")
WORK_DIR.mkdir(parents=True, exist_ok=True)

Device: cuda, Decord: GPU


In [ ]:
dl_dir = WORK_DIR / "download"

for folder in FOLDERS:
    print(f"Downloading {folder}...")
    snapshot_download(
        REPO_ID,
        repo_type="dataset",
        local_dir=str(dl_dir),
        allow_patterns=f"{folder}/*",
    )

all_videos = []
for folder in FOLDERS:
    folder_path = dl_dir / folder
    if not folder_path.exists():
        print(f"  {folder} not found, skipping")
        continue
    for f in folder_path.iterdir():
        if f.suffix.lower() in VIDEO_EXTS:
            all_videos.append(f)

print(f"\nTotal videos: {len(all_videos)}")

Fetching ... files: 0it [00:00, ?it/s]

Fetching ... files: 0it [00:00, ?it/s]


Total videos: 6000


In [ ]:
import random
sample_vids = random.sample(all_videos, min(20, len(all_videos)))

In [ ]:
from decord import cpu
for v in sample_vids[:5]:
    try:
        vr = VideoReader(str(v), ctx=cpu(0))
        print(f"{v.name}: {len(vr)} frames")
    except Exception as e:
        print(f"{v.name}: {e}")


real_94469.mp4: 266 frames
real_1086.mp4: 348 frames
real_63482.mp4: 440 frames
real_56807.mp4: 304 frames
real_50382.mp4: 248 frames


In [ ]:
import subprocess
from tqdm import tqdm

processed_dir = WORK_DIR / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

failed = 0
for v in tqdm(all_videos, desc="cutting to 10s"):
    dst = processed_dir / v.name
    if dst.exists():
        continue
    try:
        subprocess.run([
            "ffmpeg", "-y", "-i", str(v),
            "-t", "10", "-c:v", "libx264", "-an",
            "-loglevel", "error", str(dst)
        ], check=True, timeout=30)
    except Exception:
        failed += 1

shutil.rmtree(dl_dir, ignore_errors=True)

all_videos = sorted(processed_dir.glob("*.mp4"))
print(f"Processed: {len(all_videos)}, failed: {failed}")


Streaming output truncated to the last 5000 lines.
cutting to 10s: 100%|██████████| 6000/6000 [38:53<00:00,  2.57it/s]


Processed: 6000, failed: 0


In [ ]:
dec_ctx = cpu(0)

In [ ]:
def hist_corr_gpu(frames_tensor):
    gray = (frames_tensor.float() @ torch.tensor([0.299, 0.587, 0.114], device=frames_tensor.device))

    N = gray.shape[0]
    flat = gray.reshape(N, -1)
    binned = (flat / 256.0 * HIST_BINS).long().clamp(0, HIST_BINS - 1)

    hists = torch.zeros(N, HIST_BINS, device=frames_tensor.device)
    for i in range(N):
        hists[i] = torch.bincount(binned[i], minlength=HIST_BINS).float()

    hists = hists / (hists.sum(dim=1, keepdim=True) + 1e-8)

    corrs = []
    for i in range(N - 1):
        h1 = hists[i] - hists[i].mean()
        h2 = hists[i + 1] - hists[i + 1].mean()
        num = (h1 * h2).sum()
        den = torch.sqrt((h1 ** 2).sum() * (h2 ** 2).sum()) + 1e-8
        corrs.append((num / den).item())

    return corrs


def compute_cut_score(video_path):
    try:
        vr = VideoReader(str(video_path), ctx=dec_ctx)
    except Exception:
        return None

    total = len(vr)
    if total < MIN_FRAMES:
        return None

    indices = np.linspace(0, total - 1, min(SAMPLE_FRAMES, total)).astype(int).tolist()

    try:
        frames = vr.get_batch(indices)
    except Exception:
        return None

    frames_t = torch.tensor(frames.asnumpy(), device=device, dtype=torch.uint8)
    corrs = hist_corr_gpu(frames_t)

    if not corrs:
        return None

    n_cuts = sum(1 for c in corrs if c < CUT_THRESHOLD)
    return (n_cuts, len(corrs), total)


scores = []
skipped = 0

for vpath in tqdm(all_videos, desc="scoring videos (GPU)"):
    res = compute_cut_score(vpath)
    if res is None:
        skipped += 1
        continue
    n_cuts, n_compared, total = res
    cut_rate = n_cuts / n_compared
    scores.append((vpath, n_cuts, cut_rate, total))

print(f"\nScored {len(scores)} videos, skipped {skipped}")
rates = [s[2] for s in scores]
print(f"Cut rate — mean: {np.mean(rates):.3f}, median: {np.median(rates):.3f}")
print(f"  0 cuts: {sum(1 for r in rates if r == 0)}")
print(f"  >50% cuts: {sum(1 for r in rates if r > 0.5)}")

scoring videos (GPU): 100%|██████████| 6000/6000 [03:51<00:00, 25.90it/s]


Scored 6000 videos, skipped 0
Cut rate — mean: 0.040, median: 0.000
  0 cuts: 3342
  >50% cuts: 4


In [ ]:
scores.sort(key=lambda x: (x[2], x[1]))

selected = scores[:TARGET]
print(f"Selected {len(selected)} cleanest videos")
print(f"  cut_rate range: {selected[0][2]:.3f} - {selected[-1][2]:.3f}")
print(f"  max cuts in selected: {max(s[1] for s in selected)}")

rejected = scores[TARGET:]
if rejected:
    print(f"\nRejected {len(rejected)} videos")
    print(f"  cut_rate range: {rejected[0][2]:.3f} - {rejected[-1][2]:.3f}")

Selected 4000 cleanest videos
  cut_rate range: 0.000 - 0.034
  max cuts in selected: 1

Rejected 2000 videos
  cut_rate range: 0.034 - 0.621


In [ ]:
out_dir = WORK_DIR / "upload"
out_dir.mkdir(parents=True, exist_ok=True)

for vpath, n_cuts, cut_rate, total in tqdm(selected, desc="copying"):
    dst = out_dir / vpath.name
    if dst.exists():
        dst = out_dir / f"{vpath.parent.name}_{vpath.name}"
    shutil.copy2(vpath, dst)

print(f"Ready: {len(list(out_dir.iterdir()))} videos")

shutil.rmtree(dl_dir, ignore_errors=True)
print("Download dir deleted.")

copying: 100%|██████████| 4000/4000 [00:00<00:00, 6908.34it/s]

Ready: 4000 videos
Download dir deleted.


In [ ]:
files = list(out_dir.iterdir())
print(f"Uploading {len(files)} videos to {UPLOAD_PATH}...")
upload_folder(
    folder_path=str(out_dir),
    path_in_repo=UPLOAD_PATH,
    repo_id=REPO_ID,
    repo_type="dataset",
)
print("Done!")

shutil.rmtree(WORK_DIR, ignore_errors=True)

It seems you are trying to upload a large folder at once. This might take some time and then fail if the folder is too large. For such cases, it is recommended to upload in smaller batches or to use `HfApi().upload_large_folder(...)`/`hf upload-large-folder` instead. For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/upload#upload-a-large-folder.


Uploading 4000 videos to real/GenVideo-Real-clean-3k...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...er/upload/real_100000.mp4:   1%|          | 4.76kB /  607kB            

...

  ...ter/upload/real_10139.mp4:   1%|          | 4.97kB /  634kB            

  ...ter/upload/real_10171.mp4:   1%|          | 4.17kB /  531kB            

  ...ter/upload/real_10175.mp4:   1%|          | 4.63kB /  591kB            

  ...ter/upload/real_10205.mp4:   1%|          | 5.12kB /  652kB            

Done!


In [ ]:
!pip install -q transformers pillow decord

In [ ]:
import torch
from transformers import CLIPProcessor, CLIPModel
from decord import VideoReader, cpu
from pathlib import Path
from tqdm import tqdm
import numpy as np

model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").cuda()
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

REAL_TEXTS = ["a real photo", "a natural image", "a real video frame", "a photograph of a real scene"]
FAKE_TEXTS = ["a cartoon", "an animated scene", "an illustration", "an anime frame", "a drawing", "computer graphics"]

all_texts = REAL_TEXTS + FAKE_TEXTS
n_real = len(REAL_TEXTS)

def is_natural_video(video_path, n_frames=6, threshold=0.5):
    try:
        vr = VideoReader(str(video_path), ctx=cpu(0))
    except:
        return False

    indices = np.linspace(0, len(vr)-1, n_frames).astype(int).tolist()
    frames = vr.get_batch(indices).asnumpy() 

    real_scores = []
    for frame in frames:
        inputs = processor(text=all_texts, images=frame, return_tensors="pt", padding=True).to("cuda")
        with torch.no_grad():
            outputs = model(**inputs)
            probs = outputs.logits_per_image.softmax(dim=-1)[0].cpu().numpy()

        real_score = probs[:n_real].sum()
        real_scores.append(real_score)

    return np.mean(real_scores) > threshold


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

In [ ]:
from huggingface_hub import snapshot_download, upload_folder

dl_dir = Path("/content/genvideo_clip")
snapshot_download(
    "ironiss/PhysVidDetect-v1",
    repo_type="dataset",
    local_dir=str(dl_dir),
    allow_patterns=["real/GenVideo-Real-clean-3k/*"],
)

videos = sorted((dl_dir / "real/GenVideo-Real-clean-3k").glob("*.mp4"))
print(f"Total: {len(videos)}")

natural = []
cartoon = []
for v in tqdm(videos, desc="CLIP filtering"):
    if is_natural_video(v):
        natural.append(v)
    else:
        cartoon.append(v)

print(f"Natural: {len(natural)}, Cartoon/synthetic: {len(cartoon)}")


Fetching ... files: 0it [00:00, ?it/s]

Total: 4000



CLIP filtering: 100%|██████████| 4000/4000 [05:10<00:00, 12.90it/s]

Natural: 3138, Cartoon/synthetic: 862


In [ ]:
print("Sample cartoons detected:")
for v in cartoon[:10]:
    print(f"  {v.name}")

Sample cartoons detected:
  real_10062.mp4
  real_10070.mp4
  real_10348.mp4
  real_10441.mp4
  real_10473.mp4
  real_10709.mp4
  real_10939.mp4
  real_10973.mp4
  real_11002.mp4
  real_11164.mp4


In [ ]:
print(len(natural))

3138


In [ ]:
from huggingface_hub import HfApi
from huggingface_hub import CommitOperationDelete

api = HfApi()
natural_names = {v.name for v in natural}

all_hf = [f.rfilename for f in api.list_repo_tree(
    "ironiss/PhysVidDetect-v1",
    path_in_repo="real/GenVideo-Real-clean-3k",
    repo_type="dataset"
)]

to_delete = [f for f in all_hf if f.split("/")[-1] not in natural_names]
print(f"Deleting {len(to_delete)} cartoon files from HF...")


operations = [CommitOperationDelete(path_in_repo=f) for f in to_delete]
api.create_commit(
    repo_id="ironiss/PhysVidDetect-v1",
    repo_type="dataset",
    operations=operations,
    commit_message="Remove cartoon/animation videos from GenVideo-Real-clean-3k",
)
print("Done!")


Deleting 862 cartoon files from HF...
Done!
